In [1]:
%cd ..

/home/jovyan/project/film-recommendation


In [ ]:
#!/usr/bin/env python3
"""
Targeted hyperparameter tuning for recommendation models.
Performs light tuning for Collaborative Filtering (to avoid excessive runtime),
full tuning for Content-Based, Hybrid, and Cascading Hybrid.
Best parameters are saved incrementally to "best_hyperparameters.json".
If a model's best parameters already exist in the file, its optimization is skipped.
"""
import asyncio
import json
import sys
import time
import warnings
from pathlib import Path
from typing import Dict, Any, Tuple, Optional, List

import numpy as np
import pandas as pd
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

from src.data_processing.data_loader import MovieLensDataLoader
from src.data_processing.splitters import DataSplitter
from src.models.content_based import ContentBasedRecommender, ContentBasedConfig
from src.models.collaborative_filtering import CollaborativeFiltering
from src.models.hybrid import HybridRecommender
from src.models.cascading_hybrid import CascadingHybridRecommender
from src.evaluation.evaluator import RecommendationEvaluator

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------
DATASET = "ml-latest-small"              # change to "ml-latest" for large dataset
CACHE_METADATA = "data/processed/movie_metadata.parquet"
STUDY_DB = "sqlite:///optuna_studies.db" # Optuna storage for persistence
BEST_PARAMS_FILE = "best_hyperparameters.json"
VAL_SIZE = 0.2                           # fraction of train users for validation
RANDOM_SEED = 42
METRIC = "ndcg"
K_OPT = 10
NEGATIVE_SAMPLES = 99
TOP_K_EVAL = 20

# Number of trials per model (reduced for CF)
TRIALS_CB = 80
TRIALS_CF = 20                           # light tuning
TRIALS_HYBRID = 80
TRIALS_CASCADE = 80

# ----------------------------------------------------------------------
# Data loading and splitting (cached to avoid repeated I/O)
# ----------------------------------------------------------------------
async def load_movielens() -> Tuple[pd.DataFrame, pd.DataFrame]:
    loader = MovieLensDataLoader(DATASET, cache_file=CACHE_METADATA)
    data_dict = loader.load_data()
    await loader.letterboxd_data_async(max_concurrent_requests=50)

    movies_df = pd.DataFrame(loader.movie_data)
    genre_features = loader.preprocess_movies()
    movies_df = pd.concat([movies_df, genre_features], axis=1)
    movies_df = movies_df.dropna().reset_index(drop=True)

    ratings_df = data_dict["ratings"]
    return movies_df, ratings_df


def create_train_val_splits(ratings_df: pd.DataFrame, val_frac: float, seed: int):
    splitter = DataSplitter(ratings_df)
    splits = splitter.leave_one_out()
    train_full = splits["train"]
    test_full = splits["test"]

    rng = np.random.default_rng(seed)
    all_users = train_full["userId"].unique()
    n_val_users = max(1, int(len(all_users) * val_frac))
    val_users = rng.choice(all_users, size=n_val_users, replace=False)
    val_mask = train_full["userId"].isin(val_users)
    train_sub = train_full[~val_mask].copy()
    val_sub = train_full[val_mask].copy()
    return train_sub, val_sub, test_full


# ----------------------------------------------------------------------
# Evaluation helper
# ----------------------------------------------------------------------
def evaluate_model_on_val(
    model: Any,
    model_name: str,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    item_universe: List[int],
    k_values: List[int] = [K_OPT],
    max_rec: int = TOP_K_EVAL,
) -> float:
    evaluator = RecommendationEvaluator(
        models={model_name: model},
        train_df=train_df,
        test_df=val_df,
        relevance_threshold=4.0,
        user_sample_size=None,
        random_state=RANDOM_SEED,
        item_universe=item_universe,
        n_negative_samples=NEGATIVE_SAMPLES,
    )
    res_df = evaluator.evaluate_all_models(k_values=k_values, max_recommendations=max_rec)
    if res_df.empty:
        return 0.0
    row = res_df[(res_df["model"] == model_name) & (res_df["k"] == K_OPT)]
    if row.empty:
        return 0.0
    return row.iloc[0][METRIC]


# ----------------------------------------------------------------------
# Objective functions for Optuna (reduced CF search)
# ----------------------------------------------------------------------
def objective_cf_light(trial: optuna.Trial, train_df, val_df, item_universe) -> float:
    # Only tune the most impactful parameters, keep others fixed at reasonable defaults
    params = {
        "k_components": 110,                      # fixed
        "reg_all": trial.suggest_float("reg_all", 0.005, 0.2, log=True),
        "lr_all": 0.005,                          # fixed
        "n_epochs": trial.suggest_int("n_epochs", 8, 20),
        "alpha": 0.5,                             # fixed
        "min_ratings": 5,                         # fixed
    }
    model = CollaborativeFiltering(random_state=RANDOM_SEED, **params)
    model.fit(train_df)
    return evaluate_model_on_val(model, "CF", train_df, val_df, item_universe)


def objective_cb(trial: optuna.Trial, train_df, val_df, movies_df, item_universe) -> float:
    cfg = ContentBasedConfig(
        main_actor_weight=trial.suggest_float("main_actor_weight", 0.05, 0.5),
        director_weight=trial.suggest_float("director_weight", 0.02, 0.3),
        cast_weight=trial.suggest_float("cast_weight", 0.05, 0.4),
        keywords_weight=trial.suggest_float("keywords_weight", 0.1, 0.5),
        genre_weight=trial.suggest_float("genre_weight", 0.05, 0.3),
        numerical_weight=trial.suggest_float("numerical_weight", 0.0, 0.2),
        tfidf_sublinear_tf=trial.suggest_categorical("tfidf_sublinear_tf", [True, False]),
        tfidf_max_features=trial.suggest_int("tfidf_max_features", 3000, 12000, step=1000),
        similarity_threshold=trial.suggest_float("similarity_threshold", 0.05, 0.2),
        top_k_per_item=trial.suggest_int("top_k_per_item", 30, 70, step=5),
        pop_boost_weight=trial.suggest_float("pop_boost_weight", 0.0, 0.25),
        show_progress_bars=False,
    )
    model = ContentBasedRecommender(config=cfg)
    model.fit(movies_df=movies_df, ratings_df=train_df)
    return evaluate_model_on_val(model, "CB", train_df, val_df, item_universe)


def objective_hybrid(trial: optuna.Trial, train_df, val_df, movies_df, item_universe) -> float:
    # CF part (light)
    cf = CollaborativeFiltering(
        k_components=110,
        reg_all=trial.suggest_float("cf_reg", 0.005, 0.2, log=True),
        lr_all=0.005,
        n_epochs=trial.suggest_int("cf_epochs", 8, 20),
        alpha=0.5,
        min_ratings=5,
        random_state=RANDOM_SEED,
    )
    # CB part (full)
    cb_cfg = ContentBasedConfig(
        main_actor_weight=trial.suggest_float("cb_actor", 0.05, 0.5),
        director_weight=trial.suggest_float("cb_dir", 0.02, 0.3),
        cast_weight=trial.suggest_float("cb_cast", 0.05, 0.4),
        keywords_weight=trial.suggest_float("cb_kw", 0.1, 0.5),
        genre_weight=trial.suggest_float("cb_genre", 0.05, 0.3),
        numerical_weight=trial.suggest_float("cb_num", 0.0, 0.2),
        tfidf_sublinear_tf=trial.suggest_categorical("cb_sublinear", [True, False]),
        tfidf_max_features=trial.suggest_int("cb_tfidf_max", 3000, 12000, step=1000),
        similarity_threshold=trial.suggest_float("cb_sim", 0.05, 0.2),
        top_k_per_item=trial.suggest_int("cb_topk", 30, 70, step=5),
        pop_boost_weight=trial.suggest_float("cb_pop", 0.0, 0.25),
        show_progress_bars=False,
    )
    alpha = trial.suggest_float("hybrid_alpha", 0.3, 0.9)

    cf.fit(train_df)
    cb = ContentBasedRecommender(config=cb_cfg)
    cb.fit(movies_df=movies_df, ratings_df=train_df)

    hybrid = HybridRecommender(cf_model=cf, cb_model=cb, alpha=alpha)
    hybrid.fitted(cf_model=cf, cb_model=cb, movies_df=movies_df, ratings_df=train_df)
    return evaluate_model_on_val(hybrid, "Hybrid", train_df, val_df, item_universe)


def objective_cascade(trial: optuna.Trial, train_df, val_df, movies_df, item_universe) -> float:
    cf = CollaborativeFiltering(
        k_components=110,
        reg_all=trial.suggest_float("cf_reg", 0.005, 0.2, log=True),
        lr_all=0.005,
        n_epochs=trial.suggest_int("cf_epochs", 8, 20),
        alpha=0.5,
        min_ratings=5,
        random_state=RANDOM_SEED,
    )
    cb_cfg = ContentBasedConfig(
        main_actor_weight=trial.suggest_float("cb_actor", 0.05, 0.5),
        director_weight=trial.suggest_float("cb_dir", 0.02, 0.3),
        cast_weight=trial.suggest_float("cb_cast", 0.05, 0.4),
        keywords_weight=trial.suggest_float("cb_kw", 0.1, 0.5),
        genre_weight=trial.suggest_float("cb_genre", 0.05, 0.3),
        numerical_weight=trial.suggest_float("cb_num", 0.0, 0.2),
        tfidf_sublinear_tf=trial.suggest_categorical("cb_sublinear", [True, False]),
        tfidf_max_features=trial.suggest_int("cb_tfidf_max", 3000, 12000, step=1000),
        similarity_threshold=trial.suggest_float("cb_sim", 0.05, 0.2),
        top_k_per_item=trial.suggest_int("cb_topk", 30, 70, step=5),
        pop_boost_weight=trial.suggest_float("cb_pop", 0.0, 0.25),
        show_progress_bars=False,
    )
    primary_k = trial.suggest_int("primary_k", 30, 100, step=10)

    cf.fit(train_df)
    cb = ContentBasedRecommender(config=cb_cfg)
    cb.fit(movies_df=movies_df, ratings_df=train_df)

    cascade = CascadingHybridRecommender(
        primary_model=cf,
        secondary_model=cb,
        primary_k=primary_k,
    )
    cascade.fitted(primary_model=cf, secondary_model=cb)
    return evaluate_model_on_val(cascade, "Cascade", train_df, val_df, item_universe)


# ----------------------------------------------------------------------
# Study runner with skip-if-exists logic
# ----------------------------------------------------------------------
def run_study_if_missing(
    objective,
    study_name: str,
    n_trials: int,
    existing_params: Dict[str, Any],
    direction: str = "maximize",
    storage: str = STUDY_DB,
) -> Optional[optuna.Study]:
    if study_name in existing_params:
        print(f"⏭️  {study_name}: best parameters already found, skipping optimization.")
        return None
    print(f"🔍 Optimizing {study_name} ({n_trials} trials)...")
    study = optuna.create_study(
        study_name=study_name,
        direction=direction,
        sampler=TPESampler(seed=RANDOM_SEED),
        pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=10),
        storage=storage,
        load_if_exists=True,
    )
    # If the study already has completed trials (e.g., from a previous partial run),
    # Optuna will continue from where it left off. We'll just optimize for the remaining.
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    return study


def load_existing_params(filepath: Path) -> Dict[str, Any]:
    if filepath.exists():
        with open(filepath, "r") as f:
            return json.load(f)
    return {}


def save_best_params(filepath: Path, params: Dict[str, Any]):
    # Load existing, update with new, save
    existing = load_existing_params(filepath)
    existing.update(params)
    with open(filepath, "w") as f:
        json.dump(existing, f, indent=2)
    print(f"💾 Best parameters saved to {filepath}")


# ----------------------------------------------------------------------
# Main orchestration
# ----------------------------------------------------------------------
async def main():
    print("=== Loading data ===")
    movies_df, ratings_df = await load_movielens()
    print(f"Movies: {movies_df.shape}, Ratings: {ratings_df.shape}")

    train_df, val_df, test_df = create_train_val_splits(ratings_df, VAL_SIZE, RANDOM_SEED)
    item_universe = sorted(set(movies_df["movieId"].unique()))

    # Load any previously found best parameters
    best_file = Path(BEST_PARAMS_FILE)
    existing_best = load_existing_params(best_file)

    # Define the models to optimize with their objective functions and trial counts
    models_to_tune = [
        ("CF", objective_cf_light, TRIALS_CF),
        ("CB", objective_cb, TRIALS_CB),
        ("Hybrid", objective_hybrid, TRIALS_HYBRID),
        ("Cascade", objective_cascade, TRIALS_CASCADE),
    ]

    new_best_entries = {}

    for name, obj_fn, n_trials in models_to_tune:
        # Prepare the objective lambda that captures the necessary data
        # (all models need train_df, val_df, item_universe; CB/Hybrid/Cascade also need movies_df)
        if name == "CF":
            objective = lambda trial: obj_fn(trial, train_df, val_df, item_universe)
        elif name == "CB":
            objective = lambda trial: obj_fn(trial, train_df, val_df, movies_df, item_universe)
        elif name == "Hybrid":
            objective = lambda trial: obj_fn(trial, train_df, val_df, movies_df, item_universe)
        elif name == "Cascade":
            objective = lambda trial: obj_fn(trial, train_df, val_df, movies_df, item_universe)

        study = run_study_if_missing(objective, name, n_trials, existing_best)
        if study is not None:
            best_val = study.best_value
            best_params = study.best_trial.params
            new_best_entries[name] = {"params": best_params, "value": best_val}
            print(f"✅ {name}: best {METRIC}@{K_OPT} = {best_val:.4f}")
            # Save incrementally so that partial progress is not lost
            save_best_params(best_file, new_best_entries)
        else:
            # If skipped, still ensure existing entry is kept
            new_best_entries[name] = existing_best[name]

    # Final save (just in case)
    save_best_params(best_file, new_best_entries)
    print("🏁 All optimizations completed or skipped.")


await main()

INFO:src.data_processing.data_loader:Loading MovieLens dataset...
INFO:src.data_processing.data_loader:Loading existing data from cache: data/processed/movie_metadata.parquet


=== Loading data ===


INFO:src.data_processing.data_loader:Found 112 missing movies with valid TMDB IDs. Fetching...
INFO:src.data_processing.data_loader:DOWNLOAD SUMMARY: Requested: 112 | Saved: 0 | Failed: 112


Movies: (9701, 30), Ratings: (100836, 4)
🔍 Optimizing CF (20 trials)...


[I 2026-07-01 15:48:35,512] A new study created in RDB with name: CF


...

2026-07-01 15:54:09.698 | WARNING  | src.evaluation.evaluator:evaluate_model:292 - Batch error for Hybrid: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all(). Falling back to single mode.


Hybrid:   0%|          | 0/122 [00:00<?, ?it/s]

2026-07-01 15:54:09.882 | INFO     | src.evaluation.evaluator:evaluate_model:438 - 'Hybrid' done in 0.2s
2026-07-01 15:54:10.059 | INFO     | src.models.collaborative_filtering:__init__:94 - CollaborativeFiltering initialised. Explicit-feedback biased ALS model ready for training.


[I 2026-07-01 15:54:09,948] Trial 14 finished with value: 0.8795347261254838 and parameters: {'cf_reg': 0.024107513628301786, 'cf_epochs': 9, 'cb_actor': 0.3331978997134169, 'cb_dir': 0.19513781844730338, 'cb_cast': 0.24843930952943236, 'cb_kw': 0.1812699712112874, 'cb_genre': 0.09102504444871296, 'cb_num': 0.04265666712622443, 'cb_sublinear': False, 'cb_tfidf_max': 7000, 'cb_sim': 0.09219709200247717, 'cb_topk': 60, 'cb_pop': 0.10902495839307001, 'hybrid_alpha': 0.5565739722715264}. Best is trial 0 with value: 0.8795347261254838.



ALS epochs: 100%|██████████| 12/12 [00:01<00:00,  7.57it/s]
--- Step: Training complete ---
  Wall clock time : 1.63 s
  CPU time (user) : 29.93 s
  Memory RSS      : 599.6 MB
  Memory VMS      : 11907.1 MB
  CPU usage       : 0.0 %
2026-07-01 15:54:12.031 | INFO     | src.models.collaborative_filtering:fit:175 - Fitting complete | wall=1.63s cpu=29.96s mem=599.6MB
2026-07-01 15:54:12.033 | INFO     | src.models.content_based:fit:608 - Starting model fitting
2026-07-01 15:54:12.035 | INFO     | src.models.content_based:fit:615 - Cleaning text columns
2026-07-01 15:54:12.060 | INFO     | src.models.content_based:fit:619 - Building cast and keyword strings
2026-07-01 15:54:12.064 | INFO     | src.models.content_based:fit:628 - Preprocessing numerical and rating features
2026-07-01 15:54:12.065 | INFO     | src.models.content_based:_preprocess_numerical_features:122 - Preprocessing numerical features
2026-07-01 15:54:12.067 | DEBUG    | src.models.content_based:_fit_scaler:108 - Clipping